In [0]:
## Changing data types
from pyspark.sql.types import *

silver_schema = StructType([
    StructField("event_dates", DateType(), True),
    StructField("event_distance_length", StringType(), True),
    StructField("athlete_performance", StringType(), True),
    StructField("athlete_year_of_birth", IntegerType(), True),
    StructField("athlete_average_speed", DoubleType(), True),
    StructField("athlete_id", IntegerType(), True)
])

In [0]:
## Read data
bronze_data = spark.table("marathos.bronze.marathon_data")

In [0]:
## Check how many rows before cleaning
bronze_data.count()

In [0]:
## Filter out invalid events and performances
from pyspark.sql import functions

# If event has unit km or mi then performance should be in h
# If event has unit h then performance should be in km
# Else it gets filtered out

clean_event_performance = bronze_data.filter(
    (
        (functions.col("event_distance_length").contains("km") | functions.col("event_distance_length").contains("mi"))
        &
        functions.col("athlete_performance").contains("h")
    )
    |
    (
        functions.col("event_distance_length").contains("h")
        &
        functions.col("athlete_performance").contains("km")
    )
)

# If event has d (days) in athlete performance it gets filtered out
clean_data = clean_event_performance.filter(
    ~functions.col("athlete_performance").contains("d")

)


In [0]:
## Check how many rows were removed
clean_data.count()

In [0]:
from pyspark.sql import functions

# Split athlete_performance into hours, minutes, and seconds. And remove the "h"
time_units = functions.split(
    functions.trim(
        functions.split(functions.col("athlete_performance"), "h").getItem(0)
    ),
    ":"
)

# Add a column with the time in seconds
clean_data = clean_data.withColumn(
    "performance_seconds",
    time_units.getItem(0).try_cast("int") * 3600
    + time_units.getItem(1).try_cast("int") * 60
    + time_units.getItem(2).try_cast("int")
)

In [0]:
## Add event_id using dense_rank()
from pyspark.sql.window import Window

# Both event_name and event_dates are included so the same event on a different date has its own ID
w = Window.orderBy("event_name", "event_dates")

clean_data = clean_data.withColumn("event_id", functions.dense_rank().over(w))

In [0]:
## Genie Code bug fix for error: trying to cast data with km into INT

# Only convert time-based performances to seconds
# Distance-based performances (km) are kept as-is
time_based = clean_data.filter(
    functions.col("athlete_performance").contains("h")
)

# Split time for time-based performances
time_units_filtered = functions.split(
    functions.trim(
        functions.split(functions.col("athlete_performance"), "h").getItem(0)
    ),
    ":"
)

clean_data_with_seconds = time_based.withColumn(
    "performance_seconds",
    time_units_filtered.getItem(0).try_cast("int") * 3600
    + time_units_filtered.getItem(1).try_cast("int") * 60
    + time_units_filtered.getItem(2).try_cast("int")
)

# For distance-based performances, keep original value without conversion
clean_data_distance = clean_data.filter(
    functions.col("athlete_performance").contains("km")
).withColumn("performance_seconds", functions.lit(None).cast("int"))

# Union both datasets
clean_data = clean_data_with_seconds.union(clean_data_distance)

In [0]:
## Write cleaned data into silver layer
clean_data.write.format("delta").mode("overwrite").saveAsTable("marathos.silver.marathon_obt")